In [19]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Load the RFM data we saved
df = pd.read_csv(r'C:\Users\Janhavi\ChurnProject\data\processed\telco_rfm.csv')

# Create a SQLite database file (it creates the file automatically)
conn = sqlite3.connect(r'C:\Users\Janhavi\ChurnProject\data\telco.db')

# Push the dataframe into a SQL table called 'customers'
df.to_sql('customers', conn, if_exists='replace', index=False)

print("✅ Database created and data loaded!")
print(f"Total rows in database: {pd.read_sql('SELECT COUNT(*) as total FROM customers', conn).iloc[0,0]}")


✅ Database created and data loaded!
Total rows in database: 7032


In [21]:
print("Overall Churn Rate")
query = """
SELECT 
    Churn,
    COUNT(*) as customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) as percentage
FROM customers
GROUP BY Churn
"""

result = pd.read_sql(query, conn)
print(result)


Overall Churn Rate
   Churn  customer_count  percentage
0      0            5163       73.42
1      1            1869       26.58


In [23]:
print("Churn Rate by Contract Type")
query = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(Churn) as churned,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 1) as churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
"""

result = pd.read_sql(query, conn)
print(result)


Churn Rate by Contract Type
         Contract  total_customers  churned  churn_rate_pct
0  Month-to-month             3875     1655            42.7
1        One year             1472      166            11.3
2        Two year             1685       48             2.8


In [25]:
print("Revenue Lost to Churn")
query = """
SELECT
    ROUND(SUM(MonthlyCharges), 2) as total_monthly_revenue,
    ROUND(SUM(CASE WHEN Churn = 1 THEN MonthlyCharges ELSE 0 END), 2) as revenue_lost_monthly,
    ROUND(SUM(CASE WHEN Churn = 1 THEN MonthlyCharges * 12 ELSE 0 END), 2) as revenue_lost_annually
FROM customers
"""

result = pd.read_sql(query, conn)
print(result)


Revenue Lost to Churn
   total_monthly_revenue  revenue_lost_monthly  revenue_lost_annually
0               455661.0             139130.85              1669570.2


In [27]:
print("Churn by Customer Segment (RFM)")
query = """
SELECT
    Segment,
    COUNT(*) as total_customers,
    SUM(Churn) as churned,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 1) as churn_rate_pct
FROM customers
GROUP BY Segment
ORDER BY churn_rate_pct DESC
"""

result = pd.read_sql(query, conn)
print(result)


Churn by Customer Segment (RFM)
   Segment  total_customers  churned  avg_monthly_charges  churn_rate_pct
0    Loyal             4412     1382                58.63            31.3
1      VIP             1929      455                94.76            23.6
2  At-Risk              691       32                20.55             4.6


In [29]:
print("Top 10 High-Value At-Risk Customers")
query = """
SELECT 
    tenure,
    MonthlyCharges,
    TotalCharges,
    Contract,
    Segment,
    Churn
FROM customers
WHERE Churn = 1 AND MonthlyCharges > 70
ORDER BY TotalCharges DESC
LIMIT 10
"""

result = pd.read_sql(query, conn)
print(result)




Top 10 High-Value At-Risk Customers
   tenure  MonthlyCharges  TotalCharges  Contract Segment  Churn
0      72          117.80       8684.80  One year     VIP      1
1      70          115.55       8127.60  One year     VIP      1
2      72          109.25       8109.80  One year     VIP      1
3      70          115.65       7968.85  One year     VIP      1
4      68          113.15       7856.00  Two year     VIP      1
5      67          118.35       7804.15  One year     VIP      1
6      67          116.20       7752.30  Two year     VIP      1
7      70          114.20       7723.90  Two year     VIP      1
8      71          106.00       7723.70  Two year     VIP      1
9      71          108.60       7690.90  Two year     VIP      1


In [31]:
# Save all query results to one Excel file with multiple sheets
with pd.ExcelWriter(r'C:\Users\Janhavi\ChurnProject\reports\sql_analysis.xlsx', engine='openpyxl') as writer:
    
    pd.read_sql("SELECT Contract, COUNT(*) as total, SUM(Churn) as churned, ROUND(SUM(Churn)*100.0/COUNT(*),1) as churn_rate FROM customers GROUP BY Contract ORDER BY churn_rate DESC", conn).to_excel(writer, sheet_name='By Contract', index=False)
    
    pd.read_sql("SELECT Segment, COUNT(*) as total, SUM(Churn) as churned, ROUND(SUM(Churn)*100.0/COUNT(*),1) as churn_rate FROM customers GROUP BY Segment ORDER BY churn_rate DESC", conn).to_excel(writer, sheet_name='By Segment', index=False)
    
    pd.read_sql("SELECT InternetService, COUNT(*) as total, SUM(Churn) as churned, ROUND(SUM(Churn)*100.0/COUNT(*),1) as churn_rate FROM customers GROUP BY InternetService ORDER BY churn_rate DESC", conn).to_excel(writer, sheet_name='By Internet', index=False)

print("✅ Excel report saved!")

# Close the database connection
conn.close()
print("✅ Database connection closed.")


✅ Excel report saved!
✅ Database connection closed.
